# Dunnhumby 후보상품별 N/V 적합도 M1~M5 6-arm 개발 screen
historical CLV를 `q_N`, `q_V`, `q_C=percentile(n_u×v_u)`로 고정하고, 상품에는 학습기간 구매자 활동 맥락 `b_N`과 구매금액 위치 `p_V`만 부여합니다. 공유 적합도 `F(u,i)`를 M2 표현, M3 관측엣지 재배분, M4 양성행 가중에 각각 사용합니다. seed 42, DAY 1~683 학습 → 684~690 개발평가, 100 epoch, 양성당 균등 음성 1개 BPR입니다. final test·holdout은 만들지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess

REVIEWED_SHA = '0fd2e6d39de99b70258b54e113fe0bdecfb9ccb0'
REPO = Path('/content/clv-m2-lightgcn-runner')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', REVIEWED_SHA], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
%cd /content/clv-m2-lightgcn-runner

In [ ]:
import json
import torch
import lightgcn_clv_candidate_nv_fit_factorial as screen

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert screen.CODE_VERSION == 'm5-candidate-specific-nv-fit-factorial-development-screen-v1'
cfg = screen.configure_candidate_nv_fit_screen()
summary = screen.preflight_summary(cfg)
assert cfg.seed == 42 and cfg.negative_count == 1 and cfg.epochs == 100
assert cfg.rho == 0.15 and cfg.beta_m3 == 0.15 and cfg.positive_weight_lambda == 0.5
assert len(summary['trained_models']) == 6 and summary['reused_models'] == []
assert summary['fixed']['new_item_task'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = screen.run_candidate_nv_fit_screen(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

core = [
    'model_id', 'm2_expression', 'm3_edge_weight', 'm4_positive_weight',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10', 'vndcg@10',
    'coverage@10', 'top10_share@10'
]
print('1) M1~M5 6-arm 핵심 절대지표')
show(result_df[core])
print('2) 전체 비교표')
show(result_df.attrs['comparison'])
print('3) Top-10 변경 비율')
show(result_df.attrs['top10_overlap'])
print('4) ID 점수 대비 N/V 표현 점수')
show(result_df.attrs['score_diagnostics'])
print('5) 작동 진단')
print(json.dumps(result_df.attrs['mechanism_diagnostics'], ensure_ascii=False, indent=2))
print('6) 사전 판독 범위')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))